In [1]:
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

#  Copy dataset from Drive to /content for faster access
import shutil
import os

GDRIVE_DATA_PATH = "/content/drive/MyDrive/Final_Dataset"
PROJECT_DATA_MOUNT_PATH = "/content/Final_Dataset"

# Copy to Colab's local storage
if not os.path.exists(PROJECT_DATA_MOUNT_PATH):
    print(" Copying dataset from Google Drive to Colab...")
    shutil.copytree(GDRIVE_DATA_PATH, PROJECT_DATA_MOUNT_PATH)
else:
    print("Dataset already copied to Colab session storage.")

# Output directory for saving results (local)
PROJECT_OUTPUT_MOUNT_PATH = "/content/YOLO_Output"
os.makedirs(PROJECT_OUTPUT_MOUNT_PATH, exist_ok=True)

print(f"Dataset copied to: {PROJECT_DATA_MOUNT_PATH}")
!ls {PROJECT_DATA_MOUNT_PATH}


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Install YOLO Dependencies and Clone Ultralytics
print("Cloning Ultralytics YOLO repository")
%cd /content/ # Go to content directory first
!git clone https://github.com/ultralytics/ultralytics.git

print("\nInstalling Ultralytics requirements...")
%cd /content/ultralytics # Change directory into the cloned repo
!pip install ultralytics
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 # Ensure CUDA-enabled torch for Colab

print("\nUltralytics setup complete!")

In [ ]:
# Imports
from ultralytics import YOLO
import os
import datetime
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
from glob import glob
from PIL import Image as PILImage
import pandas as pd

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(" Using device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")


In [ ]:
# Make sure we are in the Ultralytics directory for YOLO commands
%cd /content/ultralytics

# Define the mounted paths (same as in previous cells)
PROJECT_DATA_MOUNT_PATH = "/content/Final_Dataset"
PROJECT_OUTPUT_MOUNT_PATH = "/content/YOLO_Output"

DATA_YAML = os.path.join(PROJECT_DATA_MOUNT_PATH, "data.yaml")# data.yaml is directly in Final_Dataset

RUN_NAME = "fire-detection-exp1" # Keep your run name
YOLO_SAVE_DIR = os.path.join(PROJECT_OUTPUT_MOUNT_PATH, "runs", "detect")

print(f"\nDATA_YAML path set to: {DATA_YAML}")
print(f"Training results will be saved to: {YOLO_SAVE_DIR}/{RUN_NAME}")

# Load pretrained model
model = YOLO('yolov8m.pt')
print("YOLOv8m model loaded.")


In [ ]:
# # TRAINING
# print("\nStarting YOLO Training...")
# results = model.train(
#     data=DATA_YAML,
#     epochs=150,
#     imgsz=640,
#     batch=32,
#     device=0, # GPU index
#     lr0=0.002,
#     lrf=0.2,
#     hsv_h=0.015,
#     hsv_s=0.7,
#     hsv_v=0.4,
#     mosaic=1.0,
#     mixup=0.2,
#     copy_paste=0.1,
#     val=True,
#     patience=20,
#     name=RUN_NAME,
#     project=YOLO_SAVE_DIR
# )
# print("\nTraining complete!")
# TRAINING
print("\nStarting YOLO Training...")
results = model.train(
    data=DATA_YAML,
    epochs=115,
    imgsz=640,
    batch=32,
    workers=8,
    device=0,
    lr0=0.002,
    lrf=0.01,
    # --- Color Augmentations (Fine-tuned from defaults) ---
    hsv_h=0.015,          # Hue: Small adjustment, fire color is distinct
    hsv_s=0.7,            # Saturation: Fairly strong, for varied lighting
    hsv_v=0.4,            # Value (Brightness): Fairly strong, for varied lighting
    # --- Geometric Augmentations (Crucial for robustness) ---
    degrees=10.0,         # Random rotation up to +/- 10 degrees
    translate=0.1,        # Random translation up to +/- 10%
    scale=0.5,            # Random scaling (0.5 = 50% to 150%) - VERY IMPORTANT for fire size variation
    shear=0.0,            # Keep at 0 unless specific need, can distort
    perspective=0.0002,   # Small random perspective transform
    flipud=0.0,           # Vertical flip (unlikely for fire, unless looking down)
    fliplr=0.5,           # Horizontal flip (fire is symmetrical enough)
    # --- Advanced Augmentations (Powerful for object detection) ---
    mosaic=1.0,           # Mosaic (mixes 4 images) - Highly effective
    mixup=0.2,            # MixUp (blends 2 images) - Good regularization
    copy_paste=0.1,       # Copy-Paste objects - Good for novel contexts/occlusions
    # --- Other important training parameters ---
    val=True,
    patience=20,
    name=RUN_NAME,
    project=YOLO_SAVE_DIR
)
print("\nTraining complete!")

In [ ]:
# 1. Source: current training folder
RUN_OUTPUT_PATH = os.path.join(YOLO_SAVE_DIR, RUN_NAME)
src_run_dir = RUN_OUTPUT_PATH  # e.g., /content/YOLO_Output/runs/detect/fire-detection-exp1

# 2. Timestamped backup name
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_folder_name = f"backup_fire-detection-exp1_{timestamp}"

# 3. Final backup destination
drive_root = "/content/drive/MyDrive/Yolo_Training_Results"
backup_dir = os.path.join(drive_root, backup_folder_name)

# 4. Create directory if not exists (automatically)
os.makedirs(drive_root, exist_ok=True)

# 5. Copy entire training run folder
print(f"\n Backing up full training run to: {backup_dir} ...")
shutil.copytree(src_run_dir, backup_dir)
print(" Backup complete!")

In [ ]:
# --- Post-Training: Display Metrics and Logs from the GDrive mount ---
RUN_OUTPUT_PATH = os.path.join(YOLO_SAVE_DIR, RUN_NAME)

# METRICS & LOSS CURVES (Image)
metrics_path = os.path.join(RUN_OUTPUT_PATH, "results.png")
if os.path.exists(metrics_path):
    print(f"\nDisplaying metrics image from: {metrics_path}")
    display(Image(filename=metrics_path))
else:
    print(f"Training metrics image not found at {metrics_path}!")

In [ ]:
# YAML TRAIN CONFIG LOG
opt_path = os.path.join(RUN_OUTPUT_PATH, "opt.yaml")
if os.path.exists(opt_path):
    print("\nTraining Config:")
    !cat {opt_path}
else:
    print(f"Training config file not found at {opt_path}!")


In [ ]:
# BEST MODEL PATH
best_model_path = os.path.join(RUN_OUTPUT_PATH, "weights", "best.pt")
print("\nBest model saved at:", best_model_path)

# VALIDATION using BEST model
print("\nRunning validation on the best model...")
# Ensure the best.pt file has been synced/is available through the mount before loading
best_model = YOLO(best_model_path) # Load from the GDrive mounted path
val_results = best_model.val()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = "/content/YOLO_Output/runs/detect/fire-detection-exp1/results.csv"

if os.path.exists(log_path):
    print(f"\nPlotting metrics from CSV: {log_path}")
    df = pd.read_csv(log_path)
    df.columns = df.columns.str.strip()  # Clean up column names

    # Plot training losses
    df[['train/box_loss', 'train/cls_loss', 'train/dfl_loss']].plot(figsize=(10, 5), title='Training Losses')
    plt.grid(True)
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.show()

    # Plot validation losses
    df[['val/box_loss', 'val/cls_loss', 'val/dfl_loss']].plot(figsize=(10, 5), title='Validation Losses')
    plt.grid(True)
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.show()

    # Plot mAP and other metrics
    df[['metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)']].plot(figsize=(10, 5), title='Validation Metrics')
    plt.grid(True)
    plt.ylabel("Score")
    plt.xlabel("Epoch")
    plt.show()

else:
    print(f"CSV log not found at {log_path}!")


In [ ]:

# INFERENCE — Side-by-side results (Adjust test_imgs path)
print("\nRunning Inference on test images (adjust 'test_imgs' path)...")
# IMPORTANT: Adjust this path to point to a test images folder within your mounted dataset
test_imgs_dir = os.path.join(PROJECT_DATA_MOUNT_PATH, "images", "test") # Example: assuming test images are in Final_Dataset/images/test
# Check if the directory exists before globbing
if os.path.exists(test_imgs_dir):
    test_imgs = glob(os.path.join(test_imgs_dir, "*.jpg"))[:5] # Take up to 5 images
    if not test_imgs:
        print(f"No .jpg images found in {test_imgs_dir}. Please check path and image types.")
    for img_path in test_imgs:
        print(f"Processing image: {img_path}")
        results = best_model(img_path)
        out_path = f"/content/pred_{os.path.basename(img_path)}" # Save predictions locally in Colab for display
        results[0].save(filename=out_path)

        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(PILImage.open(img_path))
        ax[0].set_title("Input Image")
        ax[0].axis('off')

        ax[1].imshow(PILImage.open(out_path))
        ax[1].set_title("Prediction")
        ax[1].axis('off')

        plt.tight_layout()
        plt.show()
        # Clean up local prediction file if desired
        !rm {out_path}
else:
    print(f"Test images directory not found at: {test_imgs_dir}. Please create or adjust the path.")



In [ ]:
# --- SAVE BEST & LAST MODEL TO GOOGLE DRIVE ---
import shutil

# Google Drive save path (change if needed)
gdrive_output_path = "/content/drive/MyDrive/Yolo_Training_Results/fire-detection-exp1"
os.makedirs(gdrive_output_path, exist_ok=True)

# Define source model paths
best_model_src = os.path.join(RUN_OUTPUT_PATH, "weights", "best.pt")
last_model_src = os.path.join(RUN_OUTPUT_PATH, "weights", "last.pt")

# Define destination paths
best_model_dst = os.path.join(gdrive_output_path, "best.pt")
last_model_dst = os.path.join(gdrive_output_path, "last.pt")

# Copy models
print(f"\nSaving best.pt to: {best_model_dst}")
shutil.copy2(best_model_src, best_model_dst)

print(f"Saving last.pt to: {last_model_dst}")
shutil.copy2(last_model_src, last_model_dst)

print(" Both best and last model files saved to Google Drive.")


In [ ]:
#  EXPORT MODEL (Optional - for deployment)
# best_model.export(format="onnx")  # other formats: "torchscript", "engine", etc.
